# Bölüm 1: Veriye Giriş
**Haydar Kılıç — Yapay Zeka Mühendisliği**

---
Bu notebook, Bölüm 1 ders sunumundaki teorik kavramları Python ile uygulamalı olarak göstermektedir.

**İçerik:**
1. Örnek Çalışma: Kronik Yorgunluk Sendromu
2. Veri Matrisi ve Değişken Türleri
3. Değişkenler Arasındaki İlişkiler
4. Örnekleme Yanlılığı
5. Örnekleme Stratejileri
6. Rastgele Atama ve Çıkarım

In [ ]:
# Gerekli kütüphaneler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Görsel ayarlar
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_style('whitegrid')

print('Kütüphaneler başarıyla yüklendi.')

---
## 1. Örnek Çalışma: Kronik Yorgunluk Sendromu

> **Kaynak:** Deale et al. (1997). *Cognitive behavior therapy for chronic fatigue syndrome: A randomized controlled trial.* The American Journal of Psychiatry, 154(3).

**Çalışma Tasarımı:**
- 142 hasta yönlendirildi, 60'ı dahil edildi (82'si dışlandı)
- Tedavi grubu (n=30): Bilişsel davranışçı terapi
- Kontrol grubu (n=30): Gevşeme teknikleri
- 6 aylık takipte 7 hasta çalışmadan ayrıldı (3 tedavi, 4 kontrol)

In [ ]:
# --- Çalışma verilerini oluştur ---
sonuclar = pd.DataFrame({
    'Grup'      : ['Tedavi', 'Tedavi', 'Kontrol', 'Kontrol'],
    'Sonuç'     : ['İyi', 'İyi Değil', 'İyi', 'İyi Değil'],
    'Hasta Sayısı': [19, 8, 5, 21]
})

# Özet tablo (contingency table)
pivot = sonuclar.pivot(index='Grup', columns='Sonuç', values='Hasta Sayısı')
pivot['Toplam'] = pivot.sum(axis=1)
pivot.loc['Toplam'] = pivot.sum()

print('=== 6 Aylık Takip Sonuçları ===')
print(pivot.to_string())

# Oranları hesapla
oran_tedavi = 19 / 27
oran_kontrol = 5 / 26
fark = oran_tedavi - oran_kontrol

print(f'\nTedavi grubunda iyi sonuç oranı  : {oran_tedavi:.2f}  (%{oran_tedavi*100:.0f})')
print(f'Kontrol grubunda iyi sonuç oranı : {oran_kontrol:.2f}  (%{oran_kontrol*100:.0f})')
print(f'Gruplar arası fark               : {fark:.2f}  (%{fark*100:.0f})')

In [ ]:
# --- Sonuçların görselleştirilmesi ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

renkler = {'İyi': '#2196F3', 'İyi Değil': '#BDBDBD'}

# Sol: Yığılmış çubuk grafik
ax = axes[0]
gruplar = ['Tedavi\n(n=27)', 'Kontrol\n(n=26)']
iyi      = [19/27*100, 5/26*100]
iyi_degil = [8/27*100, 21/26*100]

bars1 = ax.bar(gruplar, iyi,      color='#2196F3', label='İyi Sonuç', width=0.5)
bars2 = ax.bar(gruplar, iyi_degil, bottom=iyi, color='#BDBDBD', label='İyi Değil', width=0.5)

for bar, val in zip(bars1, iyi):
    ax.text(bar.get_x() + bar.get_width()/2, val/2,
            f'%{val:.0f}', ha='center', va='center', color='white', fontweight='bold', fontsize=13)
for bar, val, bot in zip(bars2, iyi_degil, iyi):
    ax.text(bar.get_x() + bar.get_width()/2, bot + val/2,
            f'%{val:.0f}', ha='center', va='center', color='#333', fontweight='bold', fontsize=13)

ax.set_title('İyi Sonuç Oranı (6 Ay)', fontsize=13, fontweight='bold')
ax.set_ylabel('Yüzde (%)')
ax.set_ylim(0, 110)
ax.legend(loc='upper right')

# Sağ: Hasta akış şeması
ax2 = axes[1]
ax2.axis('off')
ax2.set_title('Hasta Akışı', fontsize=13, fontweight='bold')

kutu_stili = dict(boxstyle='round,pad=0.5', facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=1.5)
disarida_stili = dict(boxstyle='round,pad=0.5', facecolor='#FFF9C4', edgecolor='#F9A825', linewidth=1.5)
tedavi_stili = dict(boxstyle='round,pad=0.5', facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=1.5)
kontrol_stili = dict(boxstyle='round,pad=0.5', facecolor='#FCE4EC', edgecolor='#880E4F', linewidth=1.5)

ax2.text(0.5, 0.92, '142 Hasta Yönlendirildi', ha='center', va='center',
         transform=ax2.transAxes, bbox=kutu_stili, fontsize=10)
ax2.annotate('', xy=(0.5, 0.78), xytext=(0.5, 0.84),
             xycoords='axes fraction', arrowprops=dict(arrowstyle='->', color='#555'))
ax2.text(0.5, 0.73, '60 Hasta Dahil Edildi', ha='center', va='center',
         transform=ax2.transAxes, bbox=kutu_stili, fontsize=10)
ax2.text(0.82, 0.86, '82 Dışlandı\n(tanı/red/başka hastalık)', ha='center', va='center',
         transform=ax2.transAxes, bbox=disarida_stili, fontsize=8.5)
ax2.annotate('', xy=(0.5, 0.59), xytext=(0.5, 0.65),
             xycoords='axes fraction', arrowprops=dict(arrowstyle='->', color='#555'))
ax2.text(0.22, 0.53, 'Tedavi Grubu\n(n=30)', ha='center', va='center',
         transform=ax2.transAxes, bbox=tedavi_stili, fontsize=10)
ax2.text(0.78, 0.53, 'Kontrol Grubu\n(n=30)', ha='center', va='center',
         transform=ax2.transAxes, bbox=kontrol_stili, fontsize=10)
ax2.text(0.5, 0.59, 'Rastgele Atama', ha='center', va='center',
         transform=ax2.transAxes, fontsize=8.5, color='#555', style='italic')
ax2.text(0.22, 0.35, '3 Ayrıldı\n→ 27 kaldı', ha='center', va='center',
         transform=ax2.transAxes, fontsize=9, color='#C62828')
ax2.text(0.78, 0.35, '4 Ayrıldı\n→ 26 kaldı', ha='center', va='center',
         transform=ax2.transAxes, fontsize=9, color='#C62828')
ax2.text(0.22, 0.18, 'İyi Sonuç: 19/27\n(%70)', ha='center', va='center',
         transform=ax2.transAxes, bbox=tedavi_stili, fontsize=10, fontweight='bold')
ax2.text(0.78, 0.18, 'İyi Sonuç: 5/26\n(%19)', ha='center', va='center',
         transform=ax2.transAxes, bbox=kontrol_stili, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.suptitle('Kronik Yorgunluk Sendromu RKÇ Sonuçları', fontsize=14,
             fontweight='bold', y=1.01)
plt.show()

In [ ]:
# --- Ki-kare testi: Fark istatistiksel olarak anlamlı mı? ---
from scipy.stats import chi2_contingency, fisher_exact

# Gözlem tablosu
gozlem = np.array([[19, 8],   # Tedavi: iyi, iyi değil
                   [5,  21]]) # Kontrol: iyi, iyi değil

chi2, p_chi2, dof, beklenen = chi2_contingency(gozlem, correction=False)
odds_ratio, p_fisher = fisher_exact(gozlem)

print('=== İstatistiksel Anlamlılık Testleri ===')
print(f'\nKi-kare testi:')
print(f'  χ² = {chi2:.3f},  df = {dof},  p = {p_chi2:.5f}')
print(f'\nFisher kesin testi (küçük örneklem için daha uygun):')
print(f'  Odds oranı = {odds_ratio:.2f},  p = {p_fisher:.5f}')

esik = 0.05
if p_chi2 < esik:
    print(f'\n→ p < {esik}: Fark istatistiksel olarak ANLAMLIDIR.')
    print('  Tedavi etkisinin tesadüften kaynaklandığı hipotezi reddedilir.')
else:
    print(f'\n→ p ≥ {esik}: Fark istatistiksel olarak ANLAMSIZDIR.')

---
## 2. Veri Matrisi ve Değişken Türleri

Sunumdaki anket verisini pandas DataFrame olarak modelleyelim ve her değişkeni türüne göre sınıflandıralım.

| Değişken | Tür |
|---|---|
| `cinsiyet` | Kategorik |
| `uyku` | Sayısal, Sürekli |
| `yatis_saati` | Kategorik, Sıralı |
| `ulkeler` | Sayısal, Kesikli |
| `korku` | Kategorik, Sıralı (sayısal da kullanılabilir) |

In [ ]:
# --- Anket verisini oluştur ---
np.random.seed(42)
n = 86  # Sunumdaki öğrenci sayısı

cinsiyet_secenekler = ['erkek', 'kadın']
ie_secenekler = ['içedönük', 'dışadönük']
saat_secenekler = ['22:00-00:00', '00:00-02:00', '02:00+']

df = pd.DataFrame({
    'cinsiyet'           : np.random.choice(cinsiyet_secenekler, n, p=[0.45, 0.55]),
    'icedönük_dışadönük' : np.random.choice(ie_secenekler, n, p=[0.4, 0.6]),
    'uyku'               : np.round(np.random.normal(6.5, 1.2, n).clip(3, 10), 1),
    'yatis_saati'        : np.random.choice(saat_secenekler, n, p=[0.35, 0.50, 0.15]),
    'ulkeler'            : np.random.randint(0, 20, n),
    'korku'              : np.random.randint(1, 6, n)
})

# Sıralı değişkeni pandas Categorical olarak tanımla
saat_sirasi = pd.CategoricalDtype(
    categories=['22:00-00:00', '00:00-02:00', '02:00+'], ordered=True)
df['yatis_saati'] = df['yatis_saati'].astype(saat_sirasi)

print('Veri Matrisinin İlk 6 Satırı:')
print(df.head(6).to_string(index=True))
print(f'\nBoyut: {df.shape[0]} gözlem × {df.shape[1]} değişken')

In [ ]:
# --- Değişken türlerini incele ---
print('=== Değişken Türleri ===')
print(f"{'Değişken':<25} {'Pandas dtype':<20} {'İstatistiksel Tür'}")
print('-' * 70)
bilgi = [
    ('cinsiyet',            'object',   'Kategorik (nominal)'),
    ('icedönük_dışadönük',  'object',   'Kategorik (nominal)'),
    ('uyku',                'float64',  'Sayısal, Sürekli'),
    ('yatis_saati',         'category', 'Kategorik, Sıralı (ordinal)'),
    ('ulkeler',             'int64',    'Sayısal, Kesikli (discrete)'),
    ('korku',               'int64',    'Kategorik, Sıralı / Sayısal'),
]
for degisken, dtype, ist_tur in bilgi:
    print(f"{degisken:<25} {dtype:<20} {ist_tur}")

In [ ]:
# --- Değişken türleri: Taksonomi diyagramı ---
fig, ax = plt.subplots(figsize=(11, 5))
ax.axis('off')
ax.set_title('Değişken Türleri Taksonomisi', fontsize=14, fontweight='bold', pad=15)

def kutu(ax, x, y, metin, genislik=0.14, yukseklik=0.10,
         renk='#E3F2FD', kenar='#1565C0', font=9.5):
    kutu_obj = mpatches.FancyBboxPatch(
        (x - genislik/2, y - yukseklik/2), genislik, yukseklik,
        boxstyle='round,pad=0.01', facecolor=renk, edgecolor=kenar, linewidth=1.5,
        transform=ax.transAxes)
    ax.add_patch(kutu_obj)
    ax.text(x, y, metin, ha='center', va='center',
            transform=ax.transAxes, fontsize=font, fontweight='bold')

def ok(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                xycoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Kök
kutu(ax, 0.50, 0.88, 'Tüm Değişkenler', genislik=0.20, renk='#1565C0', kenar='#0D47A1',
     font=10.5)
ax.texts[-1].set_color('white')

# Birinci düzey
kutu(ax, 0.27, 0.62, 'Sayısal\n(Numerical)', renk='#E8F5E9', kenar='#2E7D32')
kutu(ax, 0.73, 0.62, 'Kategorik\n(Categorical)', renk='#FFF3E0', kenar='#E65100')

ok(ax, 0.42, 0.84, 0.30, 0.68)
ok(ax, 0.58, 0.84, 0.70, 0.68)

# İkinci düzey
kutu(ax, 0.14, 0.35, 'Sürekli\n(Continuous)\nörn: uyku, boy', renk='#F1F8E9', kenar='#558B2F')
kutu(ax, 0.40, 0.35, 'Kesikli\n(Discrete)\nörn: ülke sayısı', renk='#F1F8E9', kenar='#558B2F')
kutu(ax, 0.62, 0.35, 'Nominal\n(Regular)\nörn: cinsiyet', renk='#FFF8E1', kenar='#F9A825')
kutu(ax, 0.87, 0.35, 'Sıralı\n(Ordinal)\nörn: yatış saati,\nkorku', renk='#FFF8E1', kenar='#F9A825')

ok(ax, 0.22, 0.58, 0.17, 0.41)
ok(ax, 0.30, 0.58, 0.37, 0.41)
ok(ax, 0.68, 0.58, 0.65, 0.41)
ok(ax, 0.78, 0.58, 0.84, 0.41)

plt.tight_layout()
plt.show()

In [ ]:
# --- Her değişken için özet istatistikler ---
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Anket Değişkenlerinin Dağılımları', fontsize=14, fontweight='bold')

# cinsiyet
df['cinsiyet'].value_counts().plot(kind='bar', ax=axes[0,0],
    color=['#42A5F5','#EF5350'], edgecolor='white', width=0.5)
axes[0,0].set_title('Cinsiyet (Kategorik)', fontweight='bold')
axes[0,0].set_xlabel('')
axes[0,0].tick_params(axis='x', rotation=0)

# içedönük/dışadönük
df['icedönük_dışadönük'].value_counts().plot(kind='bar', ax=axes[0,1],
    color=['#66BB6A','#FFA726'], edgecolor='white', width=0.5)
axes[0,1].set_title('İçedönük / Dışadönük (Kategorik)', fontweight='bold')
axes[0,1].set_xlabel('')
axes[0,1].tick_params(axis='x', rotation=0)

# uyku — sürekli sayısal
axes[0,2].hist(df['uyku'], bins=12, color='#26A69A', edgecolor='white')
axes[0,2].axvline(df['uyku'].mean(), color='#C62828', linestyle='--',
                  label=f"Ort: {df['uyku'].mean():.1f}")
axes[0,2].set_title('Uyku (Sayısal, Sürekli)', fontweight='bold')
axes[0,2].set_xlabel('Saat')
axes[0,2].legend()

# yatış saati — sıralı kategorik
saatler = df['yatis_saati'].value_counts().sort_index()
saatler.plot(kind='bar', ax=axes[1,0],
    color=['#7986CB','#9575CD','#BA68C8'], edgecolor='white', width=0.5)
axes[1,0].set_title('Yatış Saati (Sıralı Kategorik)', fontweight='bold')
axes[1,0].set_xlabel('')
axes[1,0].tick_params(axis='x', rotation=15)

# ülkeler — kesikli sayısal
axes[1,1].hist(df['ulkeler'], bins=15, color='#FF7043', edgecolor='white')
axes[1,1].set_title('Ziyaret Edilen Ülke (Sayısal, Kesikli)', fontweight='bold')
axes[1,1].set_xlabel('Ülke Sayısı')

# korku — sıralı kategorik / sayısal
df['korku'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1,2],
    color=['#EF9A9A','#EF5350','#E53935','#C62828','#B71C1C'],
    edgecolor='white', width=0.5)
axes[1,2].set_title('Korku (Sıralı, 1–5)', fontweight='bold')
axes[1,2].set_xlabel('Korku Düzeyi')
axes[1,2].tick_params(axis='x', rotation=0)

for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## 3. Değişkenler Arasındaki İlişkiler

**Temel Kavramlar:**
- **Açıklayıcı (bağımsız) değişken** → **Yanıt (bağımlı) değişken**
- İki değişken arasında bağlantı varsa → **İlişkili (correlated)**
- **İlişki ≠ Nedensellik** (yalnızca rastgeleştirilmiş deneyden çıkarılabilir)

Sunumdaki GPA–çalışma saati örneğini simüle edelim.

In [ ]:
# --- GPA - Çalışma saati ilişkisi ---
np.random.seed(7)
n_ogrenci = 120

calisma_saati = np.random.exponential(scale=15, size=n_ogrenci).clip(1, 70)
gurultu = np.random.normal(0, 0.25, n_ogrenci)
gpa = (3.0 + 0.012 * calisma_saati + gurultu).clip(2.8, 4.0)

# Sunumdaki aykırı değer: GPA > 4.0 olan öğrenci
calisma_saati = np.append(calisma_saati, 22)
gpa = np.append(gpa, 4.15)   # veri hatası / aykırı değer

df_gpa = pd.DataFrame({'calisma_saati': calisma_saati, 'gpa': gpa})

# Korelasyon
r, p_val = stats.pearsonr(df_gpa['calisma_saati'], df_gpa['gpa'])

fig, ax = plt.subplots(figsize=(9, 5))

# Normal noktalar
normal_mask = df_gpa['gpa'] <= 4.0
ax.scatter(df_gpa.loc[normal_mask, 'calisma_saati'],
           df_gpa.loc[normal_mask, 'gpa'],
           color='#64B5F6', alpha=0.7, s=50, label='Normal gözlem')

# Aykırı değer
ax.scatter(df_gpa.loc[~normal_mask, 'calisma_saati'],
           df_gpa.loc[~normal_mask, 'gpa'],
           color='#E53935', s=100, zorder=5,
           label='Aykırı değer (GPA > 4.0 → veri hatası?)')
ax.annotate('GPA > 4.0 ?\n(veri hatası)',
            xy=(22, 4.15), xytext=(35, 4.13),
            arrowprops=dict(arrowstyle='->', color='#E53935'),
            color='#E53935', fontsize=9)

# Regresyon çizgisi (aykırı değer hariç)
m, b = np.polyfit(df_gpa.loc[normal_mask, 'calisma_saati'],
                  df_gpa.loc[normal_mask, 'gpa'], 1)
x_line = np.linspace(1, 70, 100)
ax.plot(x_line, m*x_line + b, color='#1565C0', linewidth=2,
        linestyle='--', label=f'Regresyon doğrusu (r = {r:.2f})')

ax.axhline(4.0, color='#F9A825', linestyle=':', linewidth=1.5, label='GPA = 4.0 sınırı')
ax.set_xlabel('Haftada Çalışma Saati', fontsize=11)
ax.set_ylabel('GPA', fontsize=11)
ax.set_title('GPA ile Haftalık Çalışma Saati Arasındaki İlişki', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, 75)

print(f'Pearson korelasyon katsayısı: r = {r:.3f}, p = {p_val:.4f}')
print('Yorum: Zayıf-orta pozitif ilişki; ancak bu ilişki nedensellik KANITMAZ.')
plt.tight_layout()
plt.show()

In [ ]:
# --- Possum: Kafa uzunluğu - Kafatası genişliği ---
# Sunumdaki alıştırma örneği
try:
    import statsmodels.api as sm  # varsa veri seti için
except ImportError:
    pass

np.random.seed(21)
n_possum = 104
kafa_uzunlugu = np.random.normal(92, 3.5, n_possum).clip(84, 103)
kafatasi_genisligi = 0.55 * kafa_uzunlugu + np.random.normal(0, 1.8, n_possum)

r_p, _ = stats.pearsonr(kafa_uzunlugu, kafatasi_genisligi)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(kafa_uzunlugu, kafatasi_genisligi,
           color='#5C6BC0', alpha=0.65, s=45)
m2, b2 = np.polyfit(kafa_uzunlugu, kafatasi_genisligi, 1)
x2 = np.linspace(84, 103, 100)
ax.plot(x2, m2*x2+b2, 'r--', linewidth=1.8, label=f'r = {r_p:.2f}')
ax.set_xlabel('Kafa Uzunluğu (mm)', fontsize=11)
ax.set_ylabel('Kafatası Genişliği (mm)', fontsize=11)
ax.set_title('Possum: Kafa Uzunluğu – Kafatası Genişliği', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

# Doğru yanıt kutucuğu
ax.text(0.03, 0.95,
        'Doğru cevap (c): Pozitif ilişki\nNedensellik çıkarılamaz!',
        transform=ax.transAxes, fontsize=9,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='#FFF9C4', edgecolor='#F9A825'))

plt.tight_layout()
plt.show()

---
## 4. Örnekleme Yanlılığı

Sunumda üç temel yanlılık türü ele alınmıştır:

| Tür | Açıklama |
|---|---|
| **Yanıtsızlık** | Örneklenen kişilerin büyük çoğunluğu ankete katılmıyor |
| **Gönüllü yanıt** | Yalnızca güçlü görüşlüler yanıt veriyor |
| **Kolaylık örneklemesi** | Erişimi kolay bireyler seçiliyor |

### Tarihsel Örnek: Literary Digest 1936 Anketi

In [ ]:
# --- Literary Digest 1936: Yanlı örneklem simülasyonu ---
np.random.seed(99)

# Gerçek ABD seçmen kitlesi (1936)
# FDR: %62, Landon: %38
N_kitle = 100_000
kitle = np.random.choice(['FDR', 'Landon'], size=N_kitle, p=[0.62, 0.38])

# Literary Digest örneklemi: zengin kesim (telefon/araba sahipleri)
# Bu grup Cumhuriyetçileri daha çok destekliyordu
N_digest = 2_400_000  # yaklaşık yanıt sayısı (orantılı simüle ediyoruz)
yanli_orneklem = np.random.choice(['FDR', 'Landon'], size=10_000, p=[0.43, 0.57])

# Gerçek random örneklem
rastgele_orneklem = np.random.choice(kitle, size=10_000)

def oran_hesapla(dizi, aday):
    return (dizi == aday).mean() * 100

print('=== Literary Digest 1936 Simülasyonu ===')
print(f"\nGerçek seçim sonucu  : FDR %62 – Landon %38")
print(f"Yanıltıcı anket tahmini (Digest): FDR %{oran_hesapla(yanli_orneklem,'FDR'):.0f} – "
      f"Landon %{oran_hesapla(yanli_orneklem,'Landon'):.0f}")
print(f"Rastgele örneklem tahmini       : FDR %{oran_hesapla(rastgele_orneklem,'FDR'):.0f} – "
      f"Landon %{oran_hesapla(rastgele_orneklem,'Landon'):.0f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Yanlı Örnekleme: 1936 ABD Başkanlık Seçimi', fontsize=13, fontweight='bold')

veriler = [
    ('Gerçek Sonuç', [62, 38]),
    ('Literary Digest\n(Yanlı Örneklem)', [43, 57]),
    ('Rastgele Örneklem', [oran_hesapla(rastgele_orneklem,'FDR'),
                           oran_hesapla(rastgele_orneklem,'Landon')]),
]

renkler_fdr = ['#1565C0', '#E53935', '#2E7D32']
for ax, (baslik, oranlar), renk in zip(axes, veriler, renkler_fdr):
    adaylar = ['FDR', 'Landon']
    bar_renk = [renk, '#BDBDBD']
    bars = ax.bar(adaylar, oranlar, color=bar_renk, width=0.5, edgecolor='white')
    for bar, val in zip(bars, oranlar):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'%{val:.0f}', ha='center', fontweight='bold', fontsize=13)
    ax.set_title(baslik, fontweight='bold', fontsize=10)
    ax.set_ylim(0, 80)
    ax.set_ylabel('Oy Oranı (%)')
    ax.axhline(50, color='#555', linestyle=':', linewidth=1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("\nSonuç: 2.4 milyon anket, yanlı olduğu için küçük ama iyi bir örneklemden")
print("       daha kötü tahmin verdi. Büyük örneklem yanlılığı telafi etmez!")

---
## 5. Örnekleme Stratejileri

Üç temel örnekleme yöntemini Python ile simüle edelim:
1. **Basit Rastgele Örnekleme (BRÖ)**
2. **Tabakalı Örnekleme**
3. **Küme Örneklemesi**

In [ ]:
# --- Örnekleme Stratejileri: Görselleştirme ---
np.random.seed(2025)
N = 300  # kitle büyüklüğü
n_orneklem = 30  # örneklem büyüklüğü

# Kitleyi oluştur: 3 tabaka / 6 küme
x = np.random.uniform(0, 10, N)
y = np.random.uniform(0, 6, N)

# Tabaka ataması (y eksenine göre 3 grup)
tabaka = pd.cut(y, bins=[0, 2, 4, 6], labels=['Tabaka 1', 'Tabaka 2', 'Tabaka 3'])

# Küme ataması (ızgara)
kume_x = pd.cut(x, bins=3, labels=False)
kume_y = pd.cut(y, bins=2, labels=False)
kume = kume_x * 2 + kume_y  # 0..5 arası 6 küme

df_kitle = pd.DataFrame({'x': x, 'y': y, 'tabaka': tabaka, 'kume': kume})

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Örnekleme Stratejileri', fontsize=14, fontweight='bold')

tabaka_renkleri = {'Tabaka 1': '#90CAF9', 'Tabaka 2': '#A5D6A7', 'Tabaka 3': '#FFCC80'}
kume_renkleri = plt.cm.Set2(np.linspace(0, 1, 6))

# 1) Basit Rastgele Örnekleme
ax = axes[0]
secilen_idx = np.random.choice(N, n_orneklem, replace=False)
ax.scatter(x, y, color='#90CAF9', alpha=0.4, s=25, label='Kitle')
ax.scatter(x[secilen_idx], y[secilen_idx],
           color='#E53935', s=80, zorder=5, label='Seçilen')
ax.set_title('Basit Rastgele Örnekleme', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=9)
ax.text(0.02, 0.97, f'n = {n_orneklem}', transform=ax.transAxes,
        va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))

# 2) Tabakalı Örnekleme
ax = axes[1]
secilen_tab = []
for tab, grp in df_kitle.groupby('tabaka', observed=True):
    n_tab = max(1, round(n_orneklem * len(grp) / N))
    secilen_tab.extend(grp.sample(n_tab, random_state=42).index.tolist())

for tab, renk in tabaka_renkleri.items():
    mask = df_kitle['tabaka'] == tab
    ax.scatter(df_kitle.loc[mask, 'x'], df_kitle.loc[mask, 'y'],
               color=renk, alpha=0.45, s=25, label=tab)
    sec_mask = mask & df_kitle.index.isin(secilen_tab)
    ax.scatter(df_kitle.loc[sec_mask, 'x'], df_kitle.loc[sec_mask, 'y'],
               color='#B71C1C', s=80, zorder=5)

# Tabaka sınırlarını göster
for yval in [2, 4]:
    ax.axhline(yval, color='#555', linestyle='--', linewidth=1, alpha=0.7)
ax.set_title('Tabakalı Örnekleme', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=8, loc='upper right')

# 3) Küme Örneklemesi
ax = axes[2]
secilen_kumeler = np.random.choice(6, 3, replace=False)  # 3 küme seç

for k in range(6):
    mask = df_kitle['kume'] == k
    if k in secilen_kumeler:
        ax.scatter(df_kitle.loc[mask, 'x'], df_kitle.loc[mask, 'y'],
                   color=kume_renkleri[k], s=60, zorder=5,
                   edgecolors='#B71C1C', linewidths=1.2,
                   label=f'Küme {k+1} (seçildi)')
    else:
        ax.scatter(df_kitle.loc[mask, 'x'], df_kitle.loc[mask, 'y'],
                   color=kume_renkleri[k], alpha=0.3, s=25)

ax.set_title('Küme Örneklemesi', fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=8, loc='upper right')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# --- pandas ile gerçek tabakalı örnekleme ---
# Öğrenci verisinde cinsiyete göre tabakalı örnekleme
np.random.seed(42)

hedef_orneklem = 30  # toplam örneklem büyüklüğü

def tabakali_orneklem(df, tabaka_sutunu, n_toplam, random_state=42):
    """Orantılı tabakalı örnekleme yapar."""
    orneklem_parcalari = []
    for tabaka, grup in df.groupby(tabaka_sutunu):
        n_tabaka = round(n_toplam * len(grup) / len(df))
        n_tabaka = max(1, min(n_tabaka, len(grup)))  # sınır kontrolü
        parca = grup.sample(n=n_tabaka, random_state=random_state)
        orneklem_parcalari.append(parca)
        print(f"  {tabaka}: kitle={len(grup)}, örneklem={n_tabaka}")
    return pd.concat(orneklem_parcalari)

print('=== Cinsiyete Göre Tabakalı Örnekleme ===')
orneklem_df = tabakali_orneklem(df, 'cinsiyet', hedef_orneklem)

print(f"\nKitle cinsiyet dağılımı  : {df['cinsiyet'].value_counts().to_dict()}")
print(f"Örneklem cinsiyet dağılımı: {orneklem_df['cinsiyet'].value_counts().to_dict()}")
print(f"\nKitle oranı (kadın)   : %{df['cinsiyet'].value_counts(normalize=True)['kadın']*100:.1f}")
print(f"Örneklem oranı (kadın): %{orneklem_df['cinsiyet'].value_counts(normalize=True)['kadın']*100:.1f}")

---
## 6. Rastgele Atama ve Çıkarım

Sunumdaki 2×2 matris:

| | Rastgele Atama | Rastgele Atama YOK |
|---|---|---|
| **Rastgele Örnekleme** | Nedensellik → kitleye genelleme (**ideal deney**) | Korelasyon → kitleye genelleme |
| **Rastgele Örnekleme YOK** | Nedensellik → yalnızca örneklem | Korelasyon → yalnızca örneklem (**kötü gözlemsel**) |

In [ ]:
# --- Rastgele atamanın önemi: Karıştırıcı değişken simülasyonu ---
# Örnek: Ayakkabı boyutu ile okuma becerisi
# Gerçek neden: YAŞ — ikisini de etkiliyor (karıştırıcı / confounding variable)

np.random.seed(55)
n = 200

# Yas (5-18), her şeyi etkiliyor
yas = np.random.randint(5, 19, n)

# Ayakkabı boyutu yaşa bağlı
ayakkabi = 25 + 0.9 * yas + np.random.normal(0, 1.5, n)

# Okuma becerisi yaşa bağlı
okuma = 10 + 5 * yas + np.random.normal(0, 8, n)

r_ham, _ = stats.pearsonr(ayakkabi, okuma)

# Yaş kontrol edilince (partial correlation)
res_ayakkabi = stats.linregress(yas, ayakkabi)
res_okuma    = stats.linregress(yas, okuma)
kalinti_ayakkabi = ayakkabi - (res_ayakkabi.slope * yas + res_ayakkabi.intercept)
kalinti_okuma    = okuma    - (res_okuma.slope    * yas + res_okuma.intercept)
r_partial, _ = stats.pearsonr(kalinti_ayakkabi, kalinti_okuma)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Karıştırıcı Değişken (Confounding Variable) Örneği',
             fontsize=13, fontweight='bold')

sc = axes[0].scatter(ayakkabi, okuma, c=yas, cmap='RdYlBu_r', alpha=0.7, s=45)
plt.colorbar(sc, ax=axes[0], label='Yaş')
axes[0].set_xlabel('Ayakkabı Boyutu')
axes[0].set_ylabel('Okuma Becerisi Puanı')
axes[0].set_title(f'Ham İlişki: r = {r_ham:.2f}\n(YANILTICI — karıştırıcı: yaş)',
                  fontsize=10, fontweight='bold')

axes[1].scatter(kalinti_ayakkabi, kalinti_okuma, color='#78909C', alpha=0.6, s=45)
m_k, b_k = np.polyfit(kalinti_ayakkabi, kalinti_okuma, 1)
x_k = np.linspace(kalinti_ayakkabi.min(), kalinti_ayakkabi.max(), 100)
axes[1].plot(x_k, m_k*x_k+b_k, 'r--', linewidth=2)
axes[1].set_xlabel('Ayakkabı Boyutu (yaş etkisi çıkarılmış)')
axes[1].set_ylabel('Okuma Becerisi (yaş etkisi çıkarılmış)')
axes[1].set_title(f'Yaş Kontrol Edilince: r = {r_partial:.2f}\n(Gerçek ilişki yok!)',
                  fontsize=10, fontweight='bold')
axes[1].axhline(0, color='#555', linewidth=0.8, linestyle=':')
axes[1].axvline(0, color='#555', linewidth=0.8, linestyle=':')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print('Sonuç:')
print(f'  Ham korelasyon (yanıltıcı)  : r = {r_ham:.2f}')
print(f'  Yaş kontrol edilince (gerçek): r = {r_partial:.2f}')
print('  → İlişki, nedensellik değildir. Karıştırıcı değişken (YAŞ) her iki değişkeni birlikte artırıyordu.')

In [ ]:
# --- Özet: Rastgele Atama × Rastgele Örnekleme Matrisi ---
fig, ax = plt.subplots(figsize=(11, 6))
ax.axis('off')
ax.set_title('Rastgele Atama ve Örneklemenin Çıkarıma Etkisi',
             fontsize=14, fontweight='bold', pad=20)

def kutu2(ax, x, y, metin, alt_metin, renk, kenar, genislik=0.30, yukseklik=0.25):
    kutu_obj = mpatches.FancyBboxPatch(
        (x - genislik/2, y - yukseklik/2), genislik, yukseklik,
        boxstyle='round,pad=0.015', facecolor=renk, edgecolor=kenar,
        linewidth=2, transform=ax.transAxes)
    ax.add_patch(kutu_obj)
    ax.text(x, y + 0.04, metin, ha='center', va='center',
            transform=ax.transAxes, fontsize=10, fontweight='bold')
    ax.text(x, y - 0.06, alt_metin, ha='center', va='center',
            transform=ax.transAxes, fontsize=8.5, style='italic', color='#333')

# Başlık satırı / sütunu
ax.text(0.38, 0.90, 'Rastgele Atama VAR', ha='center', va='center',
        transform=ax.transAxes, fontsize=11, fontweight='bold', color='#1565C0')
ax.text(0.72, 0.90, 'Rastgele Atama YOK', ha='center', va='center',
        transform=ax.transAxes, fontsize=11, fontweight='bold', color='#B71C1C')
ax.text(0.05, 0.68, 'Rastgele\nÖrnekleme\nVAR', ha='center', va='center',
        transform=ax.transAxes, fontsize=10, fontweight='bold')
ax.text(0.05, 0.30, 'Rastgele\nÖrnekleme\nYOK', ha='center', va='center',
        transform=ax.transAxes, fontsize=10, fontweight='bold')

kutu2(ax, 0.38, 0.65, '✅ İdeal Deney',
      'Nedensellik sonucu\nKitleye genellenebilir',
      '#E8F5E9', '#2E7D32')
kutu2(ax, 0.72, 0.65, '⚠️ Gözlemsel Çalışma\n(İyi)',
      'Korelasyon sonucu\nKitleye genellenebilir',
      '#FFF9C4', '#F9A825')
kutu2(ax, 0.38, 0.28, '🔬 Çoğu Deney',
      'Nedensellik sonucu\nYalnızca örnekleme',
      '#E3F2FD', '#1565C0')
kutu2(ax, 0.72, 0.28, '❌ Kötü Gözlemsel',
      'Korelasyon sonucu\nYalnızca örnekleme',
      '#FFEBEE', '#C62828')

# Eksen etiketleri
ax.text(0.55, 0.97, '← Nedensellik →                      ← Korelasyon →',
        ha='center', va='center', transform=ax.transAxes, fontsize=9, color='#555')

plt.tight_layout()
plt.show()

---
## 7. Ek Örnek: Çorba Benzetmesi ile Örnekleme Kalitesi

> *"Çorba iyi karıştırılmamışsa ne kadar büyük bir kaşığınız olduğu fark etmez."*

In [ ]:
# --- Örneklem büyüklüğü vs. yanlılık karşılaştırması ---
np.random.seed(1)

# Kitle: gerçek ortalama = 50
gercek_ortalama = 50
N_kitle = 100_000
kitle_data = np.random.normal(gercek_ortalama, 10, N_kitle)

# Yanlı kitle: sadece yüksek değerlere erişiliyor (kolaylık örneklemesi)
yanli_kitle = kitle_data[kitle_data > 55]

boyutlar = [10, 50, 200, 1000, 5000, 10000]

rast_ort = []
yanli_ort = []

for n in boyutlar:
    rast_ort.append(np.random.choice(kitle_data, n).mean())
    yanli_ort.append(np.random.choice(yanli_kitle, min(n, len(yanli_kitle))).mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(boyutlar, rast_ort,   'o-', color='#1565C0', linewidth=2,
             markersize=8, label='Rastgele örnekleme (tarafsız)')
ax.semilogx(boyutlar, yanli_ort, 's--', color='#E53935', linewidth=2,
             markersize=8, label='Yanlı örnekleme (kolaylık)')
ax.axhline(gercek_ortalama, color='#2E7D32', linewidth=1.8,
            linestyle=':', label=f'Gerçek ortalama = {gercek_ortalama}')

ax.set_xlabel('Örneklem Büyüklüğü (log ölçek)', fontsize=11)
ax.set_ylabel('Örneklem Ortalaması', fontsize=11)
ax.set_title('Büyük Yanlı Örneklem vs. Küçük Tarafsız Örneklem',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print('Gözlem: Rastgele örneklem büyüdükçe gerçek değere yaklaşır.')
print('        Yanlı örneklem ne kadar büyüse de gerçek değere yaklaşamaz!')

---
## Özet

| Kavram | Tanım | Örnek |
|--------|-------|-------|
| **Veri Matrisi** | Satır = gözlem, Sütun = değişken | Anket tablosu |
| **Sayısal değişken** | Sürekli veya kesikli | uyku saati, ülke sayısı |
| **Kategorik değişken** | Nominal veya sıralı | cinsiyet, yatış saati |
| **Açıklayıcı değişken** | Etki eden taraf | Çalışma saati → GPA |
| **Yanıt değişkeni** | Etkilenen taraf | GPA |
| **Gözlemsel çalışma** | Müdahalesiz veri toplama | Anket |
| **Deney** | Rastgele atama var | RKÇ |
| **Basit RÖ** | Tamamen rastgele | Piyango |
| **Tabakalı örnekleme** | Her gruptan orantılı | Cinsiyet tabakası |
| **Küme örneklemesi** | Grubun tamamı seçilir | Sınıf örneklemesi |
| **Örnekleme yanlılığı** | Temsil sorunu | Literary Digest |
| **İlişki ≠ Nedensellik** | Karıştırıcı değişken | Ayakkabı – okuma |

---
*Notebook: Bölüm 1 — Veriye Giriş | Haydar Kılıç, Yapay Zeka Mühendisliği*